# Unit 3 Assignment: Advanced RAG System

This notebook implements a production-style Advanced RAG pipeline for AI/ML QnA:
1. Query Expansion (Multi-Query)
2. Hybrid Retrieval (BM25 + SBERT + RRF)
3. Cross-Encoder Re-Ranking
4. Final Answer Generation

In [7]:
%pip install -q rank-bm25 sentence-transformers langchain langchain-core langchain-google-genai python-dotenv numpy pandas


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True)

print("GOOGLE_API_KEY loaded:", "yes" if bool(os.getenv("GOOGLE_API_KEY")) else "no")

GOOGLE_API_KEY loaded: yes


## Part 1 - Document Corpus Setup

This corpus contains more than 10 short AI/ML documents and includes related sub-topics:
- Neural network training: gradient descent, AdamW, backpropagation
- Transformer understanding: self-attention, positional encoding
- Retrieval jargon/proper noun: BM25, Sentence-BERT, LoRA, MoE

In [9]:
corpus = [
    "Transformers use self-attention to let each token attend to other tokens in the sequence.",
    "Self-attention computes weighted sums of value vectors using query-key similarity scores.",
    "Positional encodings inject order information because attention alone is permutation-invariant.",
    "Gradient descent updates model parameters in the direction that reduces loss.",
    "AdamW combines momentum and adaptive learning rates with decoupled weight decay.",
    "Backpropagation computes gradients efficiently using the chain rule across network layers.",
    "Retrieval-Augmented Generation retrieves external context before answer generation.",
    "Cross-encoders jointly encode query and document tokens for high-precision relevance scoring.",
    "Sentence-BERT maps sentences into dense vectors for efficient semantic retrieval.",
    "BM25 is a sparse retrieval algorithm that relies on term frequency and inverse document frequency.",
    "LoRA adapts large models by learning low-rank update matrices instead of full weight updates.",
    "The paper Attention Is All You Need introduced the transformer architecture in 2017.",
    "MoE routes each token to a small subset of experts, improving parameter efficiency."
]

print(f"Corpus size: {len(corpus)} docs")

Corpus size: 13 docs


## Part 2 - Hybrid Retrieval (BM25 + SBERT + RRF)

The following cell implements the required `HybridRetriever` interface with:
- BM25 sparse ranking
- SBERT dense ranking
- Reciprocal Rank Fusion (RRF)

The `retrieve()` method returns rank contributions and fused score for analysis.

In [10]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        doc_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

        results = []
        for doc_id in range(len(self.corpus)):
            bm_rank = bm25_ranks[doc_id]
            sb_rank = sbert_ranks[doc_id]
            rrf = (1 / (self.k + bm_rank)) + (1 / (self.k + sb_rank))
            results.append({
                "doc_id": doc_id,
                "rrf_score": float(rrf),
                "bm25_rank": int(bm_rank),
                "sbert_rank": int(sb_rank),
                "text": self.corpus[doc_id],
            })

        results.sort(key=lambda x: x["rrf_score"], reverse=True)
        return results[:top_k]

hybrid = HybridRetriever(corpus)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("HybridRetriever and CrossEncoder ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8151.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9324.03it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever and CrossEncoder ready.


## Part 3 - Cross-Encoder Re-Ranker

`rerank(query, candidates, top_k)` uses `cross-encoder/ms-marco-MiniLM-L-6-v2` and scores each `(query, doc)` pair.
It preserves the original user query for reranking.

## Part 4 - Query Expansion (Option B: Multi-Query)

This notebook uses Multi-Query expansion with Gemini when available and a deterministic fallback when API is unavailable.

## Part 5 - End-to-End Pipeline

`advanced_rag(user_query)` performs:
Query Expansion -> Hybrid Retrieval -> Re-Ranking -> Final Generation

In [11]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.0)

mq_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate 3 paraphrases of the user query, one per line, no numbering."),
    ("human", "{query}")
])
mq_chain = mq_prompt | llm | StrOutputParser()

def expand_query_multi(query: str) -> list[str]:
    try:
        raw = mq_chain.invoke({"query": query})
        variants = [line.strip() for line in raw.split("\n") if line.strip()]
        if variants:
            return [query] + variants[:3]
    except Exception as err:
        print(f"Gemini unavailable for expansion; using local fallback. Reason: {type(err).__name__}")

    fallback = [
        query,
        "How is this concept implemented in transformer-based systems?",
        "What mechanism explains this behavior during model training?",
        "Technical explanation with key terms and architecture details",
    ]
    return fallback

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    pairs = [[query, c["text"]] for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    for idx, score in enumerate(ce_scores):
        candidates[idx]["cross_encoder_score"] = float(score)
    candidates.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    return candidates[:top_k]

def naive_top_doc(query: str) -> dict:
    # Dense-only retrieval with SBERT cosine
    q_vec = hybrid.sbert.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    scores = hybrid.doc_vecs @ q_vec
    top_id = int(np.argmax(scores))
    return {"doc_id": top_id, "text": corpus[top_id], "score": float(scores[top_id])}

gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the provided context. If missing, say you do not have enough information.\n\nContext:\n{context}"),
    ("human", "{question}")
])

def generate_answer(question: str, context: str) -> str:
    try:
        chain = gen_prompt | llm | StrOutputParser()
        return chain.invoke({"context": context, "question": question})
    except Exception as err:
        print(f"Gemini unavailable for generation; using extractive fallback. Reason: {type(err).__name__}")
        lines = [ln.strip() for ln in context.split("\n") if ln.strip()]
        return lines[0] if lines else "I do not have enough information to answer this."

def advanced_rag(user_query: str) -> str:
    """
    Full pipeline: Query Expansion -> Hybrid Retrieval -> Re-Ranking -> LLM Generation
    Returns the final answer string.
    """
    expanded_queries = expand_query_multi(user_query)

    union = {}
    for q in expanded_queries:
        docs = hybrid.retrieve(q, top_k=5)
        for d in docs:
            union[d["text"]] = d

    candidates = list(union.values())
    top_docs = rerank(user_query, candidates, top_k=3)
    context = "\n\n".join([d["text"] for d in top_docs])
    answer = generate_answer(user_query, context)
    return answer

print("Expansion, reranker, naive baseline, and advanced_rag() are ready.")

Expansion, reranker, naive baseline, and advanced_rag() are ready.


## Part 6 - Comparison Experiment

The next cell compares Naive RAG (dense-only top doc) vs Advanced RAG (expansion + hybrid + rerank) on three test queries and creates the required table.

In [12]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is the role of low-rank adaptation in fine-tuning?",
]

rows = []
for q in test_queries:
    naive = naive_top_doc(q)

    expanded_queries = expand_query_multi(q)
    union = {}
    for eq in expanded_queries:
        for d in hybrid.retrieve(eq, top_k=5):
            union[d["text"]] = d
    top_adv = rerank(q, list(union.values()), top_k=1)[0]

    _ = advanced_rag(q)

    rows.append({
        "Query": q,
        "Naive RAG Top Doc": naive["text"],
        "Advanced RAG Top Doc": top_adv["text"],
        "Are they different?": "Yes" if naive["text"] != top_adv["text"] else "No",
    })

comparison_df = pd.DataFrame(rows)
comparison_df

Gemini unavailable for expansion; using local fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for expansion; using local fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for generation; using extractive fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for expansion; using local fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for expansion; using local fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for generation; using extractive fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for expansion; using local fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for expansion; using local fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for generation; using extractive fallback. Reason: ChatGoogleGenerativeAIError


,Query,Naive RAG Top Doc,Advanced RAG Top Doc,Are they different?
0,how do transformers encode meaning?,Transformers use self-attention to let each to...,Transformers use self-attention to let each to...,No
1,optimization techniques for training,Gradient descent updates model parameters in t...,MoE routes each token to a small subset of exp...,Yes
2,what is the role of low-rank adaptation in fin...,LoRA adapts large models by learning low-rank ...,LoRA adapts large models by learning low-rank ...,No


## Comparison Experiment Table

| Query | Naive RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| "how do transformers encode meaning?" | Transformers use self-attention to let each token attend to other tokens in the sequence. | Transformers use self-attention to let each token attend to other tokens in the sequence. | No |
| "optimization techniques for training" | Gradient descent updates model parameters in the direction that reduces loss. | MoE routes each token to a small subset of experts, improving parameter efficiency. | Yes |
| "what is the role of low-rank adaptation in fine-tuning?" | LoRA adapts large models by learning low-rank update matrices instead of full weight updates. | LoRA adapts large models by learning low-rank update matrices instead of full weight updates. | No |

Observation: In this run, Advanced RAG differed on one query and matched Naive RAG on two queries. Gemini API calls were unavailable in this environment, so fallback expansion/generation paths were used during execution.

In [13]:
sample_question = "Explain how attention helps language models encode context."
final_answer = advanced_rag(sample_question)
print("Question:", sample_question)
print("\nAnswer:")
print(final_answer)

Gemini unavailable for expansion; using local fallback. Reason: ChatGoogleGenerativeAIError
Gemini unavailable for generation; using extractive fallback. Reason: ChatGoogleGenerativeAIError
Question: Explain how attention helps language models encode context.

Answer:
Positional encodings inject order information because attention alone is permutation-invariant.
